# Huggingface transformers

## transformers 라이브러리를 이용한 간단한 훈련 예제

In [2]:
import sys, subprocess
print(sys.executable)  # 꼭 /opt/conda/bin/python인지 확인

/opt/conda/bin/python


In [7]:
# 1) transformers 삭제
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "transformers"])

Found existing installation: transformers 5.3.0
Uninstalling transformers-5.3.0:
  Successfully uninstalled transformers-5.3.0


0

In [8]:
# !pip install transformers==4.40.0

# !pip install transformers

# 2) 최신 버전 재설치
subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"])

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)


0

In [9]:
import transformers
print(transformers.__version__)

5.3.0


In [10]:
!pip install accelerate

## 여기까지 진행 후, 설정을 열어 커널 재시작을 해주세요.

In [11]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis', framework='pt')
classifier('We are very happy to include pipeline into the transformers repository.')

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9978194236755371}]

In [12]:
# Q. 여러분들이 확인하고 싶은 문장을 넣고, 감성분석을 진행해봅시다.

sentences = [
    "I like using Hugging Face's transformers library!"
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"Sentence: {sentence}")
    print(f"Sentiment: {result['label']}, Confidence: {result['score']:.4f}")
    print()

Sentence: I like using Hugging Face's transformers library!
Sentiment: POSITIVE, Confidence: 0.5408



## Huggingface 모델 정의
### 첫 번째로는 task에 적합한 모델을 직접 선택하여 import하고, 불러오는 방식이 있습니다.
- pretrained 모델이라면 모델의 이름을 string으로,
- 직접 학습시킨 모델이라면 config와 모델을 저장한 경로를 string으로 넘겨주면 됩니다.

In [13]:
from transformers import BertForPreTraining
model = BertForPreTraining.from_pretrained('bert-base-cased')

print(model.__class__)

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

<class 'transformers.models.bert.modeling_bert.BertForPreTraining'>


### 두 번째 방법은, AutoModel을 이용

In [14]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-cased")
print(model.__class__)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'transformers.models.bert.modeling_bert.BertModel'>


## (2) Tokenizer 정의

In [15]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

### AutoTokenizer

In [16]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

### 불러온 tokenizer를 한 번 사용해볼까요?

In [17]:
encoded = tokenizer("This is Test for aiffel")
print(encoded)

{'input_ids': [101, 1188, 1110, 5960, 1111, 170, 11093, 1883, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


이 경우는 BERT의 tokenizer이기 때문에 인코딩이 된 input_ids 뿐만 아니라, token_type_ids와 attention_mask까지 모두 생성된 input 객체를 받아볼 수 있습니다.

tokenizer의 tokenize() 메서드를 사용하면 토큰 단위로 분할된 문장을 확인할 수 있습니다. 한 번 확인해볼까요?

In [18]:
# Q. 위 메서드를 사용하여 "This is Test for aiffel" 문장을 나누어 주세요.
sentence = "This is Test for aiffel"

tokens = tokenizer.tokenize(sentence)

print(tokens)

['This', 'is', 'Test', 'for', 'a', '##iff', '##el']


In [19]:
batch_sentences = ["Hello I'm a single sentence",
                    "And another sentence",
                    "And the very very last one"]

encoded_batch = tokenizer(batch_sentences)
print(encoded_batch)

{'input_ids': [[101, 8667, 146, 112, 182, 170, 1423, 5650, 102], [101, 1262, 1330, 5650, 102], [101, 1262, 1103, 1304, 1304, 1314, 1141, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1]]}


이 밖에도 tokenize할 때에 padding, truncation 등 다양한 옵션을 설정할 수 있으며, 모델이 어떤 프레임워크를 사용하는가(Tensorflow 또는 PyTorch)에 따라 input 타입을 변경 시켜주는 return_tensors 인자도 있습니다.

In [20]:
batch = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt")
# "pt"는 PyTorch의 약자, 즉, 토크나이저가 파이썬 리스트가 아니라 torch.Tensor 형태로 반환하라는 뜻

print(batch)

{'input_ids': tensor([[ 101, 8667,  146,  112,  182,  170, 1423, 5650,  102],
        [ 101, 1262, 1330, 5650,  102,    0,    0,    0,    0],
        [ 101, 1262, 1103, 1304, 1304, 1314, 1141,  102,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0]])}


## (3) Config
### Config 클래스 지정

In [21]:
from transformers import BertConfig

config = BertConfig.from_pretrained("bert-base-cased")
print(config.__class__)
print(config)

<class 'transformers.models.bert.configuration_bert.BertConfig'>
BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



In [22]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("bert-base-cased")
print(config.__class__)
print(config)

<class 'transformers.models.bert.configuration_bert.BertConfig'>
BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



두 방식으로 불러온 config의 내용에 별다른 차이가 없다는 것을 알 수 있습니다. 만약 모델을 이미 생성했다면 model.config으로 가져올 수도 있습니다.

In [23]:
model = BertForPreTraining.from_pretrained('bert-base-cased')

# Q. 생성된 모델에서 config를 가져와봅시다
config = model.config

print(config)

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



### (4) Trainer
아래 예시를 통해 이번 노드에서 살펴본 Model, Tokenizer 및 데이터셋 구성이 TrainingArguments를 통해서 Trainer에 어떻게 반영되는지 확인해 주세요.

In [24]:
## colab에서 datasets을 설치
!pip install datasets

  Using cached multiprocess-0.70.16-py312-none-any.whl.metadata (7.2 kB)
Using cached multiprocess-0.70.16-py312-none-any.whl (146 kB)
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.18
    Uninstalling multiprocess-0.70.18:
      Successfully uninstalled multiprocess-0.70.18


In [25]:
from datasets import load_dataset
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSequenceClassification

raw_datasets = load_dataset("glue", "cola")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_datasets

README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})

이번에 사용할 데이터셋은 GLUE benchmark 중 하나인 COLA dataset입니다.
데이터를 학습시킬 모델과 이들을 토큰화하기 위한 토크나이저를 불러오고, 토큰화 함수도 만들어 보겠습니다.

In [26]:
model_name_or_path = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)    # COLA dataset의 라벨은 0(unacceptable)과 1(accpetable) 두 가지로 구분됨
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

def tokenize_function(example):
    return tokenizer(example["sentence"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

trainer에 필요한 TrainingArguments를 선언

In [28]:
training_args = TrainingArguments(
    output_dir='./results',              # output이 저장될 경로
    num_train_epochs=1,              # train 시킬 총 epochs
    per_device_train_batch_size=16,  # 각 device 당 batch size
    per_device_eval_batch_size=64,   # evaluation 시에 batch size
    warmup_steps=500,                # learning rate scheduler에 따른 warmup_step 설정
    weight_decay=0.01,                 # weight decay
    logging_dir='./logs',                 # log가 저장될 경로
    do_train=True,                        # train 수행여부
    do_eval=True,                        # eval 수행여부
    eval_steps=1000,
    # group_by_length=False,   # transformers 5.3.0 에서 제외됨
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


자, 이제 위에서 완성된 내용을 바탕으로 모델 훈련과 추론을 진행해 보겠습니다.

In [30]:
# transformers v 4 일때
#
# trainer = Trainer(
#     model,                                                                    # 학습시킬 model
#     args=training_args,                                                # TrainingArguments을 통해 설정한 arguments
#     train_dataset=tokenized_datasets["train"],         # training dataset
#     eval_dataset=tokenized_datasets["validation"], # validation dataset
#     tokenizer=tokenizer,
# )

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,                                                                    # 학습시킬 model
    args=training_args,                                                # TrainingArguments을 통해 설정한 arguments
    train_dataset=tokenized_datasets["train"],         # training dataset
    eval_dataset=tokenized_datasets["validation"], # validation dataset
    data_collator=data_collator,
)

# 모델 학습
trainer.train()

Step,Training Loss
500,0.530337


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=535, training_loss=0.5266743062812591, metrics={'train_runtime': 59.1911, 'train_samples_per_second': 144.464, 'train_steps_per_second': 9.039, 'total_flos': 91092439031580.0, 'train_loss': 0.5266743062812591, 'epoch': 1.0})

In [29]:
# momory clear

import torch
import gc

# 1. 가비지 컬렉션 강제 실행
gc.collect()

# 2. PyTorch의 캐시된 메모리 해제
torch.cuda.empty_cache()

# (추가) 현재 사용 중인 텐서들을 삭제하고 싶다면
# del model, optimizer, input_data  # 변수명은 실제에 맞게 수정
# torch.cuda.empty_cache()